# Notebook 6 — 5-Fold Cross-Validation Summary**Amaç:** Notebook 2 (EfficientNetV2-S) ve Notebook 4 (Swin Transformer) modellerinin  5 katlı çapraz doğrulama sonuçlarını raporlamak.Önceki notebook'larda yalnızca **Fold 0** üzerinden değerlendirme yapılmıştı.  Bu notebook, **tüm 5 fold** için eğitim + değerlendirme yaparak  **ortalama ± standart sapma** metriklerini hesaplar.| Metrik    | Açıklama ||-----------|----------|| Accuracy  | Doğruluk || F1-Score  | Ağırlıklı F1 || Precision | Ağırlıklı Kesinlik || Recall    | Ağırlıklı Duyarlılık |

In [ ]:
!pip install timm -q

import os, random, warnings, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.model_selection import StratifiedKFold
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torch.cuda.amp import autocast, GradScaler

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
CFG = {
    'img_size'       : 224,
    'num_classes'    : 6,
    'batch_size'     : 32,
    'epochs'         : 50,
    'lr'             : 3e-4,
    'weight_decay'   : 1e-4,
    'n_folds'        : 5,
    'dropout'        : 0.2,
    'label_smoothing': 0.05,
    'unfreeze_epoch' : 5,
    'class_weights'  : [1.8862, 1.4888, 0.2547, 3.4343, 5.0, 1.8263],
    'seed'           : 42,
}

CLASS_NAMES = {
    0: 'Ant Problems', 1: 'Small Hive Beetles', 2: 'Healthy',
    3: 'Robbed Hive',  4: 'Missing Queen',      5: 'Varroa'
}

print('📋 5-Fold CV Konfigürasyonu:')
for k, v in CFG.items():
    print(f'   {k:20s}: {v}')

In [ ]:
base = Path('/kaggle/input')
DATA_DIR = None
for root, dirs, files in os.walk(base):
    for d in dirs:
        full = Path(root) / d
        try:
            subdirs = [x for x in full.iterdir() if x.is_dir()]
            if any(s.name in ['0', '1', '2'] for s in subdirs):
                DATA_DIR = full
                break
        except:
            pass
    if DATA_DIR:
        break

assert DATA_DIR is not None, 'Veri seti bulunamadı!'
print(f'✅ Veri seti: {DATA_DIR}')

records = []
for class_folder in sorted(DATA_DIR.iterdir()):
    if class_folder.is_dir():
        class_id = int(class_folder.name)
        for img_path in class_folder.glob('*'):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                records.append({'path': str(img_path), 'label': class_id})

df = pd.DataFrame(records)
print(f'📊 Toplam: {len(df)} görüntü')

# Fold assignment — aynı seed ve strateji ile tüm notebook'larla tutarlı
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)
df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(df, df['label'])):
    df.loc[val_idx, 'fold'] = fold

print(f'\n📊 Fold dağılımı:')
for fold_id in range(CFG['n_folds']):
    n = len(df[df['fold'] == fold_id])
    print(f'   Fold {fold_id}: {n} ({n/len(df)*100:.1f}%)')

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

val_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

tta_transforms = [
    val_transform,
    transforms.Compose([
        transforms.Resize((CFG['img_size'], CFG['img_size'])),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((CFG['img_size'], CFG['img_size'])),
        transforms.RandomRotation(degrees=(90, 90)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
]

print(f'✅ Augmentation pipeline hazır.')
print(f'   TTA sayısı: {len(tta_transforms)}')

In [ ]:
class BeeDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, row['label']


def train_one_epoch(model, loader, criterion, optimizer, scaler, device, scheduler=None, step_per_batch=False):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler and step_per_batch:
            scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


@torch.no_grad()
def predict_with_tta(model, df, tta_transforms, device, batch_size=32):
    model.eval()
    all_probs = []
    for tfm in tta_transforms:
        ds = BeeDataset(df, tfm)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
        probs_list = []
        for imgs, _ in loader:
            with autocast():
                logits = model(imgs.to(device))
            probs_list.append(torch.softmax(logits, dim=1).cpu().float().numpy())
        all_probs.append(np.concatenate(probs_list))
    return np.mean(all_probs, axis=0).argmax(axis=1)


print('✅ Dataset ve eğitim fonksiyonları tanımlandı.')

In [ ]:
class BeeEfficientNet(nn.Module):
    def __init__(self, num_classes=6, dropout=0.2):
        super().__init__()
        self.backbone = models.efficientnet_v2_s(weights='IMAGENET1K_V1')
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(p=dropout / 2),
            nn.Linear(512, num_classes)
        )
        self._freeze_backbone()

    def _freeze_backbone(self):
        for param in self.backbone.features.parameters():
            param.requires_grad = False

    def unfreeze_all(self):
        for param in self.backbone.features.parameters():
            param.requires_grad = True

    def forward(self, x):
        return self.backbone(x)


class BeeSwinTransformer(nn.Module):
    def __init__(self, num_classes=6, dropout=0.2,
                 model_name='swin_small_patch4_window7_224'):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        embed_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=dropout),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(p=dropout / 2),
            nn.Linear(512, num_classes)
        )
        self._freeze_backbone()

    def _freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_all(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

    def forward(self, x):
        return self.head(self.backbone(x))


print('✅ Model sınıfları tanımlandı: BeeEfficientNet, BeeSwinTransformer')

## 5-Fold Eğitim DöngüsüHer fold için model sıfırdan eğitilir, en iyi checkpoint TTA ile değerlendirilir.  Fold sonuçları toplanır ve ortalama ± std hesaplanır.

In [ ]:
def train_kfold(model_class, model_name, df, n_folds, cfg, device,
                is_swin=False):
    """
    K-Fold çapraz doğrulama eğitimi.
    Her fold için sıfırdan model oluşturur, eğitir ve TTA ile değerlendirir.
    """
    fold_results = []
    class_weights = torch.tensor(cfg['class_weights'], dtype=torch.float).to(device)

    for fold_id in range(n_folds):
        print(f'\n{"="*65}')
        print(f'  {model_name} — FOLD {fold_id}/{n_folds-1}')
        print(f'{"="*65}')

        # Train/Val split
        train_df = df[df['fold'] != fold_id].reset_index(drop=True)
        val_df   = df[df['fold'] == fold_id].reset_index(drop=True)
        print(f'  Train: {len(train_df)} | Val: {len(val_df)}')

        # DataLoaders
        train_ds = BeeDataset(train_df, train_transform)
        val_ds   = BeeDataset(val_df, val_transform)
        train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'],
                                  shuffle=True, num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_ds, batch_size=cfg['batch_size'],
                                  shuffle=False, num_workers=2, pin_memory=True)

        # Fresh model
        if is_swin:
            model = model_class(num_classes=cfg['num_classes'],
                                dropout=cfg['dropout']).to(device)
        else:
            model = model_class(num_classes=cfg['num_classes'],
                                dropout=cfg['dropout']).to(device)

        criterion = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=cfg['label_smoothing']
        )
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=cfg['lr'], weight_decay=cfg['weight_decay']
        )

        if is_swin:
            scheduler = optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=cfg['lr'],
                steps_per_epoch=len(train_loader),
                epochs=cfg['epochs'], pct_start=0.1, anneal_strategy='cos'
            )
        else:
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=cfg['unfreeze_epoch'], eta_min=1e-7
            )

        scaler = GradScaler()
        best_f1 = 0.0
        save_path = f'/kaggle/working/best_{model_name.lower().replace(" ", "_")}_fold{fold_id}.pth'

        t0 = time.time()
        for epoch in range(1, cfg['epochs'] + 1):
            # Phase 2: unfreeze backbone
            if epoch == cfg['unfreeze_epoch'] + 1:
                model.unfreeze_all()
                if is_swin:
                    optimizer = optim.AdamW([
                        {'params': model.backbone.parameters(), 'lr': cfg['lr'] * 0.1},
                        {'params': model.head.parameters(),     'lr': cfg['lr']}
                    ], weight_decay=cfg['weight_decay'])
                    scheduler = optim.lr_scheduler.CosineAnnealingLR(
                        optimizer, T_max=cfg['epochs'] - cfg['unfreeze_epoch'], eta_min=1e-7
                    )
                else:
                    optimizer = optim.AdamW([
                        {'params': model.backbone.features.parameters(), 'lr': cfg['lr'] * 0.1},
                        {'params': model.backbone.classifier.parameters(), 'lr': cfg['lr']}
                    ], weight_decay=cfg['weight_decay'])
                    scheduler = optim.lr_scheduler.CosineAnnealingLR(
                        optimizer, T_max=cfg['epochs'] - cfg['unfreeze_epoch'], eta_min=1e-7
                    )

            step_per_batch = is_swin and epoch <= cfg['unfreeze_epoch']
            train_loss, train_acc = train_one_epoch(
                model, train_loader, criterion, optimizer, scaler, device,
                scheduler if step_per_batch else None, step_per_batch
            )
            val_loss, val_acc, val_preds, val_labels = validate(
                model, val_loader, criterion, device
            )

            if not step_per_batch:
                scheduler.step()

            val_f1 = f1_score(val_labels, val_preds, average='weighted')

            if val_f1 > best_f1:
                best_f1 = val_f1
                torch.save(model.state_dict(), save_path)

            if epoch % 10 == 0 or epoch == cfg['epochs']:
                print(f'  Epoch {epoch:3d}/{cfg["epochs"]} | '
                      f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
                      f'Acc: {train_acc:.4f}/{val_acc:.4f} | '
                      f'F1: {val_f1:.4f}')

        elapsed = time.time() - t0
        print(f'  ⏱  Fold {fold_id} eğitim süresi: {elapsed/60:.1f} dk')

        # TTA evaluation with best checkpoint
        model.load_state_dict(torch.load(save_path))
        model.eval()
        tta_preds   = predict_with_tta(model, val_df, tta_transforms, device, cfg['batch_size'])
        true_labels = val_df['label'].values

        acc  = accuracy_score(true_labels, tta_preds)
        f1   = f1_score(true_labels, tta_preds, average='weighted')
        prec = precision_score(true_labels, tta_preds, average='weighted')
        rec  = recall_score(true_labels, tta_preds, average='weighted')

        fold_results.append({
            'fold': fold_id,
            'accuracy': acc * 100,
            'f1': f1 * 100,
            'precision': prec * 100,
            'recall': rec * 100,
        })

        print(f'\n  📊 Fold {fold_id} TTA Sonuçları:')
        print(f'     Accuracy  : {acc*100:.2f}%')
        print(f'     F1-Score  : {f1*100:.2f}%')
        print(f'     Precision : {prec*100:.2f}%')
        print(f'     Recall    : {rec*100:.2f}%')

        # Bellek temizliği
        del model, optimizer, scheduler, scaler, criterion
        torch.cuda.empty_cache()

    return pd.DataFrame(fold_results)


print('✅ train_kfold fonksiyonu tanımlandı.')

## EfficientNetV2-S — 5-Fold Cross-Validation

In [ ]:
print('🚀 EfficientNetV2-S 5-Fold CV başlıyor...')
effnet_results = train_kfold(
    model_class=BeeEfficientNet,
    model_name='EfficientNet',
    df=df,
    n_folds=CFG['n_folds'],
    cfg=CFG,
    device=DEVICE,
    is_swin=False
)

print('\n' + '='*65)
print('EfficientNetV2-S — 5-Fold Sonuçları')
print('='*65)
print(effnet_results.to_string(index=False))
print(f'\n📊 Ortalama ± Std:')
for col in ['accuracy', 'f1', 'precision', 'recall']:
    mean = effnet_results[col].mean()
    std  = effnet_results[col].std()
    print(f'   {col:12s}: {mean:.2f}% ± {std:.2f}%')

## Swin Transformer — 5-Fold Cross-Validation

In [ ]:
print('🚀 Swin Transformer 5-Fold CV başlıyor...')
swin_results = train_kfold(
    model_class=BeeSwinTransformer,
    model_name='Swin',
    df=df,
    n_folds=CFG['n_folds'],
    cfg=CFG,
    device=DEVICE,
    is_swin=True
)

print('\n' + '='*65)
print('Swin Transformer — 5-Fold Sonuçları')
print('='*65)
print(swin_results.to_string(index=False))
print(f'\n📊 Ortalama ± Std:')
for col in ['accuracy', 'f1', 'precision', 'recall']:
    mean = swin_results[col].mean()
    std  = swin_results[col].std()
    print(f'   {col:12s}: {mean:.2f}% ± {std:.2f}%')

## Özet Tablo — Mean ± Std (5-Fold CV)

In [ ]:
def format_mean_std(series):
    return f'{series.mean():.2f} ± {series.std():.2f}'


summary_data = []
for model_name, results_df in [('EfficientNetV2-S', effnet_results),
                                ('Swin Transformer', swin_results)]:
    summary_data.append({
        'Model': model_name,
        'Accuracy (%)': format_mean_std(results_df['accuracy']),
        'F1-Score (%)': format_mean_std(results_df['f1']),
        'Precision (%)': format_mean_std(results_df['precision']),
        'Recall (%)': format_mean_std(results_df['recall']),
    })

summary_df = pd.DataFrame(summary_data)

print('='*85)
print('5-FOLD CROSS-VALIDATION SONUÇLARI (Mean ± Std)')
print('='*85)
print(summary_df.to_string(index=False))
print('='*85)

# Fold bazlı detaylı tablo
print('\n\n📋 EfficientNetV2-S — Fold Bazlı Sonuçlar:')
print(effnet_results.to_string(index=False, float_format='%.2f'))

print('\n📋 Swin Transformer — Fold Bazlı Sonuçlar:')
print(swin_results.to_string(index=False, float_format='%.2f'))

## Görselleştirme

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('5-Fold Cross-Validation Sonuçları', fontsize=14, fontweight='bold')

folds = range(CFG['n_folds'])
x = np.arange(len(folds))
width = 0.35

# Accuracy per fold
axes[0].bar(x - width/2, effnet_results['accuracy'], width,
            label='EfficientNetV2-S', color='#3498DB', edgecolor='white')
axes[0].bar(x + width/2, swin_results['accuracy'], width,
            label='Swin Transformer', color='#E74C3C', edgecolor='white')
axes[0].set_title('Accuracy per Fold (%)')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Fold {i}' for i in folds])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.4)

# F1 per fold
axes[1].bar(x - width/2, effnet_results['f1'], width,
            label='EfficientNetV2-S', color='#3498DB', edgecolor='white')
axes[1].bar(x + width/2, swin_results['f1'], width,
            label='Swin Transformer', color='#E74C3C', edgecolor='white')
axes[1].set_title('F1-Score per Fold (%)')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('F1-Score (%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Fold {i}' for i in folds])
axes[1].legend()
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('kfold_cv_results.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('5-Fold CV — Metrik Dağılımları (Box Plot)', fontsize=14, fontweight='bold')

metrics = ['accuracy', 'f1', 'precision', 'recall']
metric_labels = ['Accuracy (%)', 'F1-Score (%)', 'Precision (%)', 'Recall (%)']

for ax, metric, label in zip(axes, metrics, metric_labels):
    data = [effnet_results[metric].values, swin_results[metric].values]
    bp = ax.boxplot(data, labels=['EfficientNet', 'Swin'], patch_artist=True,
                    widths=0.5, showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor='gold', markersize=8))
    bp['boxes'][0].set_facecolor('#3498DB')
    bp['boxes'][1].set_facecolor('#E74C3C')
    for box in bp['boxes']:
        box.set_alpha(0.7)
    ax.set_title(label)
    ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('kfold_cv_boxplot.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
print('='*70)
print('   NOTEBOOK 6 — 5-FOLD CROSS-VALIDATION SONUÇ RAPORU')
print('='*70)

print('\n📊 Özet Tablo (Mean ± Std):')
print(summary_df.to_string(index=False))

print('\n\n📋 Akademik Katkılar:')
print('   ✅ 5 katlı çapraz doğrulama ile istatistiksel güvenilirlik sağlandı')
print('   ✅ Mean ± Std ile model kararlılığı raporlandı')
print('   ✅ Hem EfficientNetV2-S hem Swin Transformer tam CV değerlendirmesi')
print('   ✅ Her fold bağımsız olarak eğitildi ve TTA ile değerlendirildi')

# CSV olarak kaydet
effnet_results.to_csv('efficientnet_kfold_results.csv', index=False)
swin_results.to_csv('swin_kfold_results.csv', index=False)
summary_df.to_csv('kfold_cv_summary.csv', index=False)
print('\n✅ Sonuçlar CSV olarak kaydedildi.')

print('\n▶  Sonuçlar makaleye eklenmeye hazır.')
print('='*70)